In [ ]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# =========================
# 1. 경로 설정
# =========================

BASE_DIR = r"D:\캡스톤"

input_file = os.path.join(BASE_DIR, "0630_베이스_1_GIS결합완료.xlsx")


# =========================
# 2. 데이터 불러오기
# =========================

df = pd.read_excel(input_file)

print("데이터 크기:", df.shape)
print(df.head())


# =========================
# 3. 계약년월 처리
# =========================
# 계약년월이 202001 같은 형태라고 가정

df["계약년월"] = df["계약년월"].astype(str)

df["계약년"] = df["계약년월"].str[:4].astype(int)
df["계약월"] = df["계약년월"].str[4:6].astype(int)


# =========================
# 4. 종속변수 설정
# =========================

target = "m2당가"


# =========================
# 5. 입력변수 설정
# =========================
# Model 1: 단지명 제외 모델

feature_cols = [
    # 지역 변수
    "자치구",
    "행정동",
    "경도",
    "위도",

    # 거래 특성
    "전용면적(㎡)",
    "층",
    "계약년",
    "계약월",

    # 단지 물리적 특성
    "경과연수",
    "공동주택_대지면적(㎡)",
    "공동주택_건축면적(㎡)",
    "공동주택_건폐율(%)",
    "공동주택_연면적(㎡)",
    "공동주택_용적률산정연면적(㎡)",
    "공동주택_용적률(%)",
    "공동주택_세대수(세대)",
    "공동주택_주건축물수",
    "공동주택_부속건축물수",
    "공동주택_부속건축물면적(㎡)",
    "세대당_주차수",
    "공동주택_총주차수",

    # GIS / 생활 인프라 변수
    "subway_nearest_dist",
    "subway_500m_count",
    "bus_stop_nearest_dist",
    "bus_stop_500m_count",
    "school_nearest_dist",
    "school_750m_count",
    "park_nearest_dist",
    "park_750m_count",
    "culture_nearest_dist",
    "culture_1000m_count",
    "commercial_nearest_dist",
    "commercial_750m_count",
    "hospital_nearest_dist",
    "hospital_750m_count"
]


# 실제 데이터에 없는 컬럼이 있는지 확인
missing_cols = [col for col in feature_cols + [target] if col not in df.columns]

if missing_cols:
    print("다음 컬럼이 데이터에 없습니다:")
    print(missing_cols)
    raise ValueError("컬럼명을 확인해주세요.")


# =========================
# 6. X, y 구성
# =========================

X = df[feature_cols].copy()
y = df[target].copy()


# =========================
# 7. 범주형 변수 원-핫 인코딩
# =========================

categorical_cols = ["자치구", "행정동"]

X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=False
)

print("인코딩 후 입력변수 개수:", X.shape[1])


# =========================
# 8. 훈련 데이터 / 테스트 데이터 분할
# =========================
# train:test = 8:2

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("훈련 데이터 크기:", X_train.shape)
print("테스트 데이터 크기:", X_test.shape)


# =========================
# 9. 랜덤포레스트 모델 학습
# =========================

rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1
)

rf_model.fit(X_train, y_train)


# =========================
# 10. 예측
# =========================

y_pred = rf_model.predict(X_test)


# =========================
# 11. 성능 평가
# =========================

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n===== RandomForest Model 1 성능 =====")
print(f"MAE  : {mae:,.4f}")
print(f"RMSE : {rmse:,.4f}")
print(f"R²   : {r2:.4f}")


# =========================
# 12. 변수 중요도 확인
# =========================

feature_importance = pd.DataFrame({
    "변수명": X.columns,
    "중요도": rf_model.feature_importances_
}).sort_values(by="중요도", ascending=False)

print("\n===== 변수 중요도 Top 30 =====")
print(feature_importance.head(30))



데이터 크기: (15230, 40)
             시군구          경도         위도   행정동                         단지명  \
0  서울특별시 강남구 대치동  127.054749  37.495867  대치1동                      대치아이파크   
1  서울특별시 강남구 청담동  127.045500  37.521695   청담동  청담2차이-편한세상(201동,202동,203동)   
2  서울특별시 강남구 수서동  127.092172  37.480856  일원본동                     강남데시앙포레   
3  서울특별시 강남구 삼성동  127.045194  37.516250  삼성2동                    롯데캐슬프레미어   
4  서울특별시 강남구 수서동  127.086602  37.485485  일원본동                        까치마을   

   전용면적(㎡)    계약년월   계약년  계약월  계약일  ...  세대당_주차대수     용적률    건폐율 거래층  아파트_브랜드  \
0  114.970  201612  2016   12   31  ...  1.524740  274.94  14.40  19     아이파크   
1  113.210  201612  2016   12   30  ...  1.528169  229.47  25.35   5       기타   
2   59.970  201612  2016   12   30  ...  1.207116  165.96  19.75   8      데시앙   
3  121.931  201612  2016   12   30  ...  1.872370  269.63  16.16  12     롯데캐슬   
4   39.600  201612  2016   12   29  ...  0.008553  208.40  21.65  13       기타   

   대지면적(㎡)    건축면적(㎡)      연면적

ValueError: 컬럼명을 확인해주세요.

In [3]:
# =========================
# Train 성능 vs Test 성능 비교
# =========================

y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_r2 = r2_score(y_train, y_train_pred)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

print("\n===== Train 성능 =====")
print(f"MAE  : {train_mae:,.4f}")
print(f"RMSE : {train_rmse:,.4f}")
print(f"R²   : {train_r2:.4f}")

print("\n===== Test 성능 =====")
print(f"MAE  : {test_mae:,.4f}")
print(f"RMSE : {test_rmse:,.4f}")
print(f"R²   : {test_r2:.4f}")


===== Train 성능 =====
MAE  : 352,653.6927
RMSE : 582,850.2108
R²   : 0.9961

===== Test 성능 =====
MAE  : 963,401.1159
RMSE : 1,672,902.7354
R²   : 0.9670


In [11]:
import os
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# =========================
# 1. 경로 설정
# =========================

BASE_DIR = r"D:\캡스톤"

input_file = os.path.join(BASE_DIR, "0630_베이스_1_GIS결합완료.xlsx")


# =========================
# 2. 데이터 불러오기
# =========================

df = pd.read_excel(input_file)

print("데이터 크기:", df.shape)
print(df.head())




# =========================
# 4. 종속변수 설정
# =========================

target = "m2당가"


# =========================
# 5. 입력변수 설정
# =========================
# Model 1: 단지명 제외 모델

feature_cols = [
    # 지역 변수
    "경도",
    "위도",

    # 거래 특성
    "전용면적(㎡)",
    "층",

    # 단지 물리적 특성
    "경과연수",
    "공동주택_건폐율(%)",
    "공동주택_용적률(%)",
    "공동주택_세대수(세대)",
    "공동주택_주건축물수",
    "공동주택_부속건축물수",
    "공동주택_부속건축물면적(㎡)",
    "세대당_주차수",
    "공동주택_총주차수"
]


# 실제 데이터에 없는 컬럼이 있는지 확인
missing_cols = [col for col in feature_cols + [target] if col not in df.columns]

if missing_cols:
    print("다음 컬럼이 데이터에 없습니다:")
    print(missing_cols)
    raise ValueError("컬럼명을 확인해주세요.")


# =========================
# 6. X, y 구성
# =========================

X = df[feature_cols].copy()
y = df[target].copy()



# =========================
# 8. 훈련 데이터 / 테스트 데이터 분할
# =========================
# train:test = 8:2

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

print("훈련 데이터 크기:", X_train.shape)
print("테스트 데이터 크기:", X_test.shape)


# =========================
# 9. 랜덤포레스트 모델 학습
# =========================

rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1
)

rf_model.fit(X_train, y_train)


# =========================
# 10. 예측
# =========================

y_pred = rf_model.predict(X_test)


# =========================
# 11. 성능 평가
# =========================

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n===== RandomForest Model 1 성능 =====")
print(f"MAE  : {mae:,.4f}")
print(f"RMSE : {rmse:,.4f}")
print(f"R²   : {r2:.4f}")


# =========================
# 12. 변수 중요도 확인
# =========================

feature_importance = pd.DataFrame({
    "변수명": X.columns,
    "중요도": rf_model.feature_importances_
}).sort_values(by="중요도", ascending=False)

print("\n===== 변수 중요도 Top 30 =====")
print(feature_importance.head(30))

데이터 크기: (16171, 37)
             시군구  자치구          경도         위도   행정동  \
0  서울특별시 강남구 대치동  강남구  127.054749  37.495867  대치1동   
1  서울특별시 강남구 청담동  강남구  127.045500  37.521695   청담동   
2  서울특별시 강남구 수서동  강남구  127.092172  37.480856  일원본동   
3  서울특별시 강남구 삼성동  강남구  127.045194  37.516250  삼성2동   
4  서울특별시 강남구 수서동  강남구  127.086602  37.485485  일원본동   

                          단지명  전용면적(㎡)    계약년월          m2당가   층  ...  \
0                      대치아이파크  114.970  201612  1.442115e+07  19  ...   
1  청담2차이-편한세상(201동,202동,203동)  113.210  201612  9.451462e+06   5  ...   
2                     강남데시앙포레   59.970  201612  1.283975e+07   8  ...   
3                    롯데캐슬프레미어  121.931  201612  1.328620e+07  12  ...   
4                        까치마을   39.600  201612  1.313131e+07  13  ...   

   school_nearest_dist  school_750m_count  park_nearest_dist  park_750m_count  \
0           268.329781                 10        1057.434641                0   
1           159.345806                  4         615.

In [12]:
# =========================
# Train 성능 vs Test 성능 비교
# =========================

y_train_pred = rf_model.predict(X_train)
y_test_pred = rf_model.predict(X_test)

train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
train_r2 = r2_score(y_train, y_train_pred)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
test_r2 = r2_score(y_test, y_test_pred)

print("\n===== Train 성능 =====")
print(f"MAE  : {train_mae:,.4f}")
print(f"RMSE : {train_rmse:,.4f}")
print(f"R²   : {train_r2:.4f}")

print("\n===== Test 성능 =====")
print(f"MAE  : {test_mae:,.4f}")
print(f"RMSE : {test_rmse:,.4f}")
print(f"R²   : {test_r2:.4f}")


===== Train 성능 =====
MAE  : 652,017.6389
RMSE : 1,023,482.5933
R²   : 0.9879

===== Test 성능 =====
MAE  : 1,492,215.4887
RMSE : 2,220,832.5137
R²   : 0.9426


In [13]:
from sklearn.model_selection import KFold, cross_validate
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# =========================
# 기본 RandomForest 모델
# =========================

rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1
)

# =========================
# 5-Fold 교차검증 설정
# =========================

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# =========================
# 평가 지표 설정
# =========================
# sklearn은 오차 지표를 음수로 반환하므로 나중에 -를 붙여서 양수로 바꿈

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

cv_results = cross_validate(
    rf_model,
    X,
    y,
    cv=kf,
    scoring=scoring,
    n_jobs=-1,
    return_train_score=True
)

# =========================
# 결과 정리
# =========================

cv_summary = pd.DataFrame({
    "Fold": range(1, 6),
    "Train_MAE": -cv_results["train_MAE"],
    "Test_MAE": -cv_results["test_MAE"],
    "Train_RMSE": -cv_results["train_RMSE"],
    "Test_RMSE": -cv_results["test_RMSE"],
    "Train_R2": cv_results["train_R2"],
    "Test_R2": cv_results["test_R2"]
})

print("\n===== 5-Fold 교차검증 결과 =====")
print(cv_summary)

print("\n===== 평균 성능 =====")
print(f"Train MAE 평균  : {cv_summary['Train_MAE'].mean():,.4f}")
print(f"Test MAE 평균   : {cv_summary['Test_MAE'].mean():,.4f}")
print(f"Train RMSE 평균 : {cv_summary['Train_RMSE'].mean():,.4f}")
print(f"Test RMSE 평균  : {cv_summary['Test_RMSE'].mean():,.4f}")
print(f"Train R² 평균   : {cv_summary['Train_R2'].mean():.4f}")
print(f"Test R² 평균    : {cv_summary['Test_R2'].mean():.4f}")

print("\n===== Test 성능 표준편차 =====")
print(f"Test MAE 표준편차  : {cv_summary['Test_MAE'].std():,.4f}")
print(f"Test RMSE 표준편차 : {cv_summary['Test_RMSE'].std():,.4f}")
print(f"Test R² 표준편차   : {cv_summary['Test_R2'].std():.4f}")


===== 5-Fold 교차검증 결과 =====
   Fold      Train_MAE      Test_MAE    Train_RMSE     Test_RMSE  Train_R2  \
0     1  655607.463407  1.453400e+06  1.027076e+06  2.188033e+06  0.987815   
1     2  658175.638454  1.471445e+06  1.039684e+06  2.197018e+06  0.987370   
2     3  662001.500757  1.430209e+06  1.043645e+06  2.137541e+06  0.987363   
3     4  657321.044365  1.446516e+06  1.016579e+06  2.191541e+06  0.988002   
4     5  652342.696862  1.466540e+06  1.036470e+06  2.198459e+06  0.987596   

    Test_R2  
0  0.943540  
1  0.945615  
2  0.947054  
3  0.944503  
4  0.942864  

===== 평균 성능 =====
Train MAE 평균  : 657,089.6688
Test MAE 평균   : 1,453,621.8993
Train RMSE 평균 : 1,032,690.7723
Test RMSE 평균  : 2,182,518.4277
Train R² 평균   : 0.9876
Test R² 평균    : 0.9447

===== Test 성능 표준편차 =====
Test MAE 표준편차  : 16,456.2956
Test RMSE 표준편차 : 25,490.9370
Test R² 표준편차   : 0.0017
